In [27]:
import mlflow
mlflow.end_run() # kill stuck run

In [28]:
import pandas as pd
import numpy as np
import os, pickle, ast, itertools
import mlflow
from scipy.sparse import coo_matrix
from gensim.models import Word2Vec
import implicit

# ── paths ──────────────────────────────────────────────────────────────────
feat = './mbdump_small/features/'
out  = './mbdump_small/models/'
os.makedirs(out, exist_ok=True)

# ── load ───────────────────────────────────────────────────────────────────
implicit_df   = pd.read_csv(f'{feat}implicit.tsv',             sep='\t')
tod_profile   = pd.read_csv(f'{feat}tod_profile.tsv',          sep='\t')
sessions_df   = pd.read_csv(f'{feat}sessions.tsv',             sep='\t')
notifications = pd.read_csv('./mbdump_small/notifications.tsv', sep='\t')
fav_artists   = pd.read_csv('./mbdump_small/fav_artists.tsv',   sep='\t')

# ══════════════════════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════════════════════
def precision_at_k(recommended, relevant, k):
    """fraction of top-k recs that are relevant"""
    return len(set(recommended[:k]) & set(relevant)) / k

def average_precision_at_k(recommended, relevant, k):
    """AP@K — rewards hitting relevant items earlier"""
    if not relevant:
        return 0.0
    hits, score = 0, 0.0
    for i, rec in enumerate(recommended[:k], 1):
        if rec in relevant:
            hits  += 1
            score += hits / i
    return score / min(len(relevant), k)

def evaluate(ids_batch, scores_batch, test_dict, rec_dec, k=10):
    """
    ids_batch    → array (n_users, k) of encoded item ids
    test_dict    → {user_enc: [recording_ids]} held-out ground truth
    """
    precisions, aps = [], []
    for uid_enc, ids in enumerate(ids_batch):
        relevant = test_dict.get(uid_enc, [])
        if not relevant:
            continue
        recommended = [rec_dec[i] for i in ids[:k]]
        precisions.append(precision_at_k(recommended, relevant, k))
        aps.append(average_precision_at_k(recommended, relevant, k))
    return {
        f'precision_at_{k}': round(np.mean(precisions), 4),
        f'MAP_at_{k}':        round(np.mean(aps), 4)
    }

# ══════════════════════════════════════════════════════════════════════════
# TRAIN / TEST SPLIT
# leave-one-out: last play per user → test, rest → train
# ══════════════════════════════════════════════════════════════════════════
implicit_df = implicit_df.sort_values(['user_id','plays'], ascending=[True, False])

test_df  = implicit_df.groupby('user_id').last().reset_index()
train_df = implicit_df.merge(
    test_df[['user_id','recording_id']],
    on=['user_id','recording_id'],
    how='left', indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

print(f"Train → {len(train_df)} | Test → {len(test_df)}")

# encode on TRAIN only
user_enc = {u: i for i, u in enumerate(train_df['user_id'].unique())}
rec_enc  = {r: i for i, r in enumerate(train_df['recording_id'].unique())}
user_dec = {i: u for u, i in user_enc.items()}
rec_dec  = {i: r for r, i in rec_enc.items()}

def build_matrix(df):
    rows = df['user_id'].map(user_enc)
    cols = df['recording_id'].map(rec_enc)
    mask = rows.notna() & cols.notna()
    rows, cols = rows[mask].astype(int), cols[mask].astype(int)
    data = df['implicit_score'].values[mask]
    item_user = coo_matrix(
        (data, (cols, rows)),
        shape=(len(rec_enc), len(user_enc))
    ).tocsr()
    return item_user, item_user.T.tocsr()

item_user_train, user_item_train = build_matrix(train_df)

# test dict: user_enc → [recording_ids held out]
test_dict = {}
for _, row in test_df.iterrows():
    uid_enc = user_enc.get(row['user_id'])
    rid     = row['recording_id']
    if uid_enc is not None and rid in rec_enc:
        test_dict.setdefault(uid_enc, []).append(rid)

# ══════════════════════════════════════════════════════════════════════════
# 1. ALS GRID SEARCH
# ══════════════════════════════════════════════════════════════════════════
als_grid = {
    'factors':       [32, 64, 128],
    'iterations':    [10, 20],
    'regularization':[0.001, 0.01, 0.1]
}

best_als        = None
best_als_score  = -1
best_als_params = {}
als_results     = []

all_user_ids = list(range(len(user_enc)))

for factors, iterations, reg in itertools.product(
        als_grid['factors'],
        als_grid['iterations'],
        als_grid['regularization']):

    with mlflow.start_run(run_name=f"ALS_f{factors}_i{iterations}_r{reg}"):
        mlflow.log_params({"factors": factors, "iterations": iterations, "regularization": reg})

        model = implicit.als.AlternatingLeastSquares(
            factors=factors, iterations=iterations,
            regularization=reg, random_state=42
        )
        model.fit(user_item_train)

        ids_batch, scores_batch = model.recommend(
            all_user_ids, user_item_train,
            N=10, filter_already_liked_items=True
        )

        metrics = evaluate(ids_batch, scores_batch, test_dict, rec_dec, k=10)
        mlflow.log_metrics(metrics)

        row = {"factors": factors, "iterations": iterations,
               "regularization": reg, **metrics}
        als_results.append(row)
        print(f"  ALS f={factors} i={iterations} r={reg} → {metrics}")

        if metrics['MAP_at_10'] > best_als_score:
            best_als_score  = metrics['MAP_at_10']
            best_als        = model
            best_als_params = row
            ids_batch_best  = ids_batch
            scores_batch_best = scores_batch

als_results_df = pd.DataFrame(als_results).sort_values('MAP_at_10', ascending=False)
print(f"\nBest ALS → {best_als_params}")

# build recs with best model
all_recs = []
for uid_enc, ids, scores in zip(all_user_ids, ids_batch_best, scores_batch_best):
    uid = user_dec[uid_enc]
    for rank, (iid, score) in enumerate(zip(ids, scores), 1):
        all_recs.append({
            'user_id':      uid,
            'recording_id': rec_dec[iid],
            'score':        round(float(score), 4),
            'rank':         rank
        })
als_recs = pd.DataFrame(all_recs)
als_recs.to_csv(f'{out}als_recommendations.tsv', sep='\t', index=False)

with open(f'{out}als_model.pkl', 'wb') as f:
    pickle.dump({'model': best_als, 'user_enc': user_enc, 'rec_enc': rec_enc,
                 'user_dec': user_dec, 'rec_dec': rec_dec,
                 'best_params': best_als_params}, f)

# ══════════════════════════════════════════════════════════════════════════
# 2. song2vec GRID SEARCH
# ══════════════════════════════════════════════════════════════════════════
sessions_df['recording_sequence'] = sessions_df['recording_sequence'].apply(
    lambda x: [str(r) for r in ast.literal_eval(x)] if isinstance(x, str) else [str(r) for r in x]
)
sequences = sessions_df['recording_sequence'].tolist()

# split sequences 80/20
split      = int(len(sequences) * 0.8)
train_seqs = sequences[:split]
test_seqs  = sequences[split:]

w2v_grid = {
    'vector_size': [32, 64, 128],
    'window':      [3, 5, 10],
    'epochs':      [5, 10]
}

def evaluate_w2v(model, test_seqs, k=5):
    """predict last song from context — hit rate@k"""
    hits = 0
    total = 0
    for seq in test_seqs:
        if len(seq) < 2:
            continue
        context = seq[:-1]
        target  = seq[-1]
        known   = [s for s in context if s in model.wv]
        if not known or target not in model.wv:
            continue
        seed_vec = np.mean([model.wv[s] for s in known], axis=0)
        similar  = [w for w, _ in model.wv.most_similar([seed_vec], topn=k)]
        hits  += int(target in similar)
        total += 1
    return round(hits / total, 4) if total else 0.0

best_w2v        = None
best_w2v_score  = -1
best_w2v_params = {}
w2v_results     = []

for vec_size, window, epochs in itertools.product(
        w2v_grid['vector_size'],
        w2v_grid['window'],
        w2v_grid['epochs']):

    with mlflow.start_run(run_name=f"w2v_v{vec_size}_w{window}_e{epochs}"):
        mlflow.log_params({"vector_size": vec_size, "window": window, "epochs": epochs})

        model = Word2Vec(
            sentences=train_seqs, vector_size=vec_size,
            window=window, min_count=1,
            workers=4, epochs=epochs, seed=42
        )
        hit_rate = evaluate_w2v(model, test_seqs, k=5)
        mlflow.log_metric("hit_rate_at_5", hit_rate)

        row = {"vector_size": vec_size, "window": window,
               "epochs": epochs, "hit_rate_at_5": hit_rate}
        w2v_results.append(row)
        print(f"  W2V v={vec_size} w={window} e={epochs} → hit_at_5={hit_rate}")

        if hit_rate > best_w2v_score:
            best_w2v_score  = hit_rate
            best_w2v        = model
            best_w2v_params = row

w2v_results_df = pd.DataFrame(w2v_results).sort_values('hit_rate_at_5', ascending=False)
print(f"\nBest W2V → {best_w2v_params}")

best_w2v.save(f'{out}song2vec.model')
vocab  = list(best_w2v.wv.index_to_key)
emb_df = pd.DataFrame(best_w2v.wv.vectors, index=vocab)
emb_df.index.name = 'recording_id'
emb_df.to_csv(f'{out}song2vec_embeddings.tsv', sep='\t')

# ══════════════════════════════════════════════════════════════════════════
# 3. TOD
# ══════════════════════════════════════════════════════════════════════════
TOP_N = 10

tod_top = (
    tod_profile
    .sort_values(['user_id','time_of_day','play_count'], ascending=[True,True,False])
    .groupby(['user_id','time_of_day'])
    .head(TOP_N)
    .reset_index(drop=True)
)
tod_top['rank'] = tod_top.groupby(['user_id','time_of_day']).cumcount() + 1
tod_top.to_csv(f'{out}tod_playlists.tsv', sep='\t', index=False)
print(f"TOD done → {tod_top['user_id'].nunique()} users × 3 slots")

# ══════════════════════════════════════════════════════════════════════════
# 4. HYBRID — grid search TOD boost weight
# ══════════════════════════════════════════════════════════════════════════
tod_set = set(zip(tod_top['user_id'], tod_top['recording_id'], tod_top['time_of_day']))

# ground truth for hybrid: test_df recording per user
hybrid_test = test_df.groupby('user_id')['recording_id'].apply(list).to_dict()

boost_grid   = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
boost_results = []

for boost in boost_grid:
    maps = []
    for slot in ['morning', 'afternoon', 'night']:
        tmp = als_recs.copy()
        tmp['time_of_day']  = slot
        tmp['hybrid_score'] = tmp['score'] + tmp.apply(
            lambda r: boost if (r['user_id'], r['recording_id'], slot) in tod_set else 0, axis=1
        )
        tmp = (tmp.sort_values(['user_id','hybrid_score'], ascending=[True,False])
                  .groupby('user_id').head(TOP_N))

        for uid, grp in tmp.groupby('user_id'):
            recommended = grp['recording_id'].tolist()
            relevant    = hybrid_test.get(uid, [])
            maps.append(average_precision_at_k(recommended, relevant, k=10))

    mean_map = round(np.mean(maps), 4)
    boost_results.append({'boost': boost, 'MAP_at_10': mean_map})
    print(f"  Hybrid boost={boost} → MAP_at_10={mean_map}")

boost_results_df = pd.DataFrame(boost_results).sort_values('MAP_at_10', ascending=False)
best_boost       = boost_results_df.iloc[0]['boost']
print(f"\nBest boost → {best_boost}")

# build hybrid with best boost
hybrid_rows = []
for slot in ['morning', 'afternoon', 'night']:
    tmp = als_recs.copy()
    tmp['time_of_day']  = slot
    tmp['tod_boost']    = tmp.apply(
        lambda r: best_boost if (r['user_id'], r['recording_id'], slot) in tod_set else 0, axis=1
    )
    tmp['hybrid_score'] = tmp['score'] + tmp['tod_boost']
    tmp = (tmp.sort_values(['user_id','hybrid_score'], ascending=[True,False])
              .groupby('user_id').head(TOP_N)
              .reset_index(drop=True))
    tmp['rank'] = tmp.groupby('user_id').cumcount() + 1
    hybrid_rows.append(tmp)

hybrid = pd.concat(hybrid_rows).reset_index(drop=True)
hybrid.to_csv(f'{out}hybrid_playlists.tsv', sep='\t', index=False)
print(f"Hybrid done → {len(hybrid)} rows")

# ══════════════════════════════════════════════════════════════════════════
# 5. Notifications
# ══════════════════════════════════════════════════════════════════════════
user_notifs = (
    notifications
    .merge(fav_artists, on=['user_id','artist_id'], how='inner')
    .sort_values(['user_id','notified_at'], ascending=[True,False])
    [['user_id','artist_id','release_id','release_name','notified_at']]
    .reset_index(drop=True)
)
user_notifs.to_csv(f'{out}user_notifications.tsv', sep='\t', index=False)
print(f"Notifications done → {len(user_notifs)} rows")

# ══════════════════════════════════════════════════════════════════════════
# 6. Excel — all outputs + tuning results
# ══════════════════════════════════════════════════════════════════════════
with pd.ExcelWriter(f'{out}recommendations_final.xlsx', engine='openpyxl') as writer:
    als_recs.to_excel(              writer, sheet_name='als_recommendations',  index=False)
    tod_top.to_excel(               writer, sheet_name='tod_playlists',         index=False)
    hybrid.to_excel(                writer, sheet_name='hybrid_playlists',      index=False)
    emb_df.reset_index().to_excel(  writer, sheet_name='song2vec_embeddings',   index=False)
    user_notifs.to_excel(           writer, sheet_name='notifications',          index=False)
    als_results_df.to_excel(        writer, sheet_name='als_tuning',             index=False)
    w2v_results_df.to_excel(        writer, sheet_name='w2v_tuning',             index=False)
    boost_results_df.to_excel(      writer, sheet_name='hybrid_tuning',          index=False)

print("Saved → recommendations_final.xlsx")

# ══════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════
def get_playlist(user_id, time_of_day, n=10):
    return (hybrid[(hybrid['user_id'] == user_id) &
                   (hybrid['time_of_day'] == time_of_day)]
            .head(n)[['rank','recording_id','hybrid_score']])

def get_notifications(user_id, n=5):
    return user_notifs[user_notifs['user_id'] == user_id].head(n)

# ══════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════
print("\n=== BEST PARAMS ===")
print(f"  ALS    → {best_als_params}")
print(f"  W2V    → {best_w2v_params}")
print(f"  Boost  → {best_boost}")

print("\n=== OUTPUTS ===")
for name, df in [('als_recs',    als_recs),
                 ('tod_top',     tod_top),
                 ('hybrid',      hybrid),
                 ('embeddings',  emb_df),
                 ('user_notifs', user_notifs)]:
    print(f"  {name:25s} → {len(df)} rows")

print("\n=== SAMPLE: user 1 morning ===")
print(get_playlist(1, 'morning'))
print("\n=== SAMPLE: user 1 notifications ===")
print(get_notifications(1))

Train → 13276 | Test → 175


  0%|          | 0/10 [00:00<?, ?it/s]

  ALS f=32 i=10 r=0.001 → {'precision_at_10': 0.0, 'MAP_at_10': 0.0}


  0%|          | 0/10 [00:00<?, ?it/s]

  ALS f=32 i=10 r=0.01 → {'precision_at_10': 0.0, 'MAP_at_10': 0.0}


...

  ALS f=128 i=20 r=0.001 → {'precision_at_10': 0.0, 'MAP_at_10': 0.0}


  0%|          | 0/20 [00:00<?, ?it/s]

  ALS f=128 i=20 r=0.01 → {'precision_at_10': 0.0, 'MAP_at_10': 0.0}


  0%|          | 0/20 [00:00<?, ?it/s]

  ALS f=128 i=20 r=0.1 → {'precision_at_10': 0.0, 'MAP_at_10': 0.0}

Best ALS → {'factors': 32, 'iterations': 20, 'regularization': 0.001, 'precision_at_10': 0.0143, 'MAP_at_10': 0.0143}
  W2V v=32 w=3 e=5 → hit_at_5=0.25
  W2V v=32 w=3 e=10 → hit_at_5=0.25
  W2V v=32 w=5 e=5 → hit_at_5=0.25
  W2V v=32 w=5 e=10 → hit_at_5=0.25
  W2V v=32 w=10 e=5 → hit_at_5=0.25
  W2V v=32 w=10 e=10 → hit_at_5=0.25
  W2V v=64 w=3 e=5 → hit_at_5=0.25
  W2V v=64 w=3 e=10 → hit_at_5=0.25
  W2V v=64 w=5 e=5 → hit_at_5=0.25
  W2V v=64 w=5 e=10 → hit_at_5=0.25
  W2V v=64 w=10 e=5 → hit_at_5=0.25
  W2V v=64 w=10 e=10 → hit_at_5=0.25
  W2V v=128 w=3 e=5 → hit_at_5=0.25
  W2V v=128 w=3 e=10 → hit_at_5=0.25
  W2V v=128 w=5 e=5 → hit_at_5=0.25
  W2V v=128 w=5 e=10 → hit_at_5=0.25
  W2V v=128 w=10 e=5 → hit_at_5=0.25
  W2V v=128 w=10 e=10 → hit_at_5=0.25

Best W2V → {'vector_size': 32, 'window': 3, 'epochs': 5, 'hit_rate_at_5': 0.25}
TOD done → 175 users × 3 slots
  Hybrid boost=0.1 → MAP_at_10=0.0006
  Hybrid boo